모델 배포 개론 08  
Last modified : 2026.03   
작성 : 박광성 (모두의연구소)  
수정 : 김지성 (모두의연구소)  

In [1]:
# ============================================================
# 0. 필수 패키지 설치 셀
# ============================================================

import sys
import subprocess

packages = [
    "uvicorn",
    "fastapi",
    "pydantic",
    "nest_asyncio",
    "requests",
    "datasets",
    "pandas",
    "numpy",
    "scikit-learn",
    "joblib",
    "matplotlib",
    "streamlit"
]

for package in packages:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        package
    ])

print("✅ 필수 패키지 설치 완료")

✅ 필수 패키지 설치 완료


In [2]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# 코드를 저장할 폴더를 미리 만들어 둡니다.
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다 (다른 노트북의 서버일 가능성).')
        print('   그 노트북에서 stop_server(8002)을 실행하거나 커널을 종료한 뒤, 이 셀을 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    # 모델 로드 때문에 기동이 느릴 수 있다 — 최대 5분 대기 (첫 실행은 다운로드 포함)
    for i in range(600):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('서버 스레드가 종료됐습니다. 위 로그를 확인하세요.')
            return server
        if i > 0 and i % 20 == 0:
            print(f'  ... 모델 로드 중 ({i//2}초 경과)')
        time.sleep(0.5)
    print('5분 내에 서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

서버 도우미 준비 완료 (serve_in_thread, stop_server)


# Day 8 — 자율 프로젝트: 에이전트형 장사비서 ML 모델 서빙 서비스 만들기

이번 노트북은 기존 Day 8 자율 프로젝트 구조를 유지하면서, **허깅페이스 커피 매출 데이터셋**을 사용해 장사비서의 매출 예측 MVP를 만들고, 여기에 **에이전트형 학습 오케스트레이터 설계**를 결합합니다.

핵심 목표는 단순히 모델을 학습하는 것이 아니라, 데이터 점검 → 학습 가능 여부 판단 → 모델 학습 → 평가 → 리포트 → 재학습 조건 판단까지 이어지는 **Agentic ML Pipeline**을 실습하는 것입니다.


> 이번 버전은 특정 외부 에이전트 런타임 연결이 아니라, **일반 에이전트가 ML 학습 과정을 어떻게 관찰·판단·리포트하는지 경험하는 교육용 노트북**입니다.


## 1. 프로젝트 요구사항

### 1.1 목표

허깅페이스의 커피 관련 매출 데이터셋을 사용해 다음 기능을 가진 장사비서 ML 서비스를 만듭니다.

1. 커피 매출 데이터를 불러온다.
2. 장사비서 표준 포맷으로 정리한다.
3. 매출 예측 모델을 학습한다.
4. 모델 성능을 평가한다.
5. 에이전트가 데이터 품질, 학습 가능 여부, 성능 개선 여부를 해석한다.
6. FastAPI로 예측 API를 만든다.
7. Streamlit으로 간단한 테스트 UI를 만든다.

### 1.1.1 이번 프로젝트의 컨셉

> 모델은 데이터를 보고 패턴을 학습하고,  
> 에이전트는 모델이 잘 학습하고 있는지 판단하고 기록하는 학습 관리자 역할을 한다.


### 1.2 제한 사항

- 데이터는 우선 허깅페이스 `tablegpt/CoffeeSales` 데이터셋을 사용합니다.
- 인터넷 또는 허깅페이스 접근이 안 되는 환경에서는 노트북이 멈추지 않도록 샘플 데이터를 자동 생성합니다.
- 초기 MVP이므로 딥러닝보다 `RandomForestRegressor` 기반의 가벼운 ML 모델을 사용합니다.
- 에이전트는 실제 LLM API 호출 없이, 규칙 기반 판단 함수와 `.md` 정책 문서로 설계합니다.
- 운영 모델 교체는 자동 승인하지 않고, 리포트 기반으로 사람이 승인하는 구조를 가정합니다.


### 1.3 평가 기준

- 데이터셋을 장사비서 표준 스키마로 변환했는가?
- 매출 예측 모델을 학습하고 저장했는가?
- FastAPI `/health`, `/predict`, `/agent/report` 엔드포인트가 동작하는가?
- 에이전트가 데이터 품질, 모델 성능, 다음 개선안을 설명하는가?
- 모델 성능 지표와 실행 결과를 MD 파일로 정리할 수 있는가?


---

## 2. 모델과 데이터 선택

이번 프로젝트는 일반적인 텍스트 분류 모델 대신, **커피 매출 데이터 → 장사비서 매출 예측 모델**로 바꿉니다.

사용 데이터셋 후보:

- `tablegpt/CoffeeSales`: 커피 판매 거래 데이터
- `jason1966/agungpambudi_trends-product-coffee-shop-sales-revenue-dataset`: 커피숍 판매/수익 데이터
- 기타 커피 리뷰/이미지 데이터셋

이번 노트북에서는 장사비서 테스트 목적에 가장 가까운 **커피 판매 데이터**를 사용합니다.


### 2.2 도메인별 추천 예시

| 도메인 | 가능한 모델/데이터 | 이번 프로젝트 선택 |
|---|---|---|
| 커피 매출 분석 | 커피 판매 데이터, POS 데이터, 메뉴별 판매량 | ✅ 선택 |
| 커피 리뷰 분석 | 리뷰 텍스트, 감성 분석 | 추후 확장 |
| 커피 원두 이미지 | 원두 이미지 분류 | 추후 확장 |
| 장사비서 | 매출 예측, 메뉴 수요 예측, 내일 할 일 추천 | ✅ 선택 |

이번 프로젝트의 핵심은 “커피 데이터셋으로 장사비서의 모델 학습 루프를 테스트할 수 있는가?”입니다.


### 2.3 데이터 로드 + 모델 동작 확인

아래 셀은 허깅페이스 데이터셋을 불러오고, 장사비서 표준 포맷으로 변환한 뒤, 간단한 매출 예측 모델을 학습합니다.

실행 전 필요 패키지:

```bash
pip install datasets pandas scikit-learn joblib fastapi uvicorn streamlit requests
```


In [3]:

# 2.3 허깅페이스 커피 데이터셋 로드 + 장사비서 표준 포맷 변환 + 기본 모델 학습

import os
import json
import math
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_DIRS = [
    "data/raw", "data/processed", "data/reports",
    "models/current", "models/candidates", "models/archive",
    "experiments", "logs", "agent", "app", "frontend"
]
for d in PROJECT_DIRS:
    os.makedirs(d, exist_ok=True)

DATASET_NAME = "tablegpt/CoffeeSales"

def make_fallback_coffee_sales(n_days: int = 120) -> pd.DataFrame:
    """허깅페이스 접근이 안 되는 환경에서도 실습이 가능하도록 샘플 커피 매출 데이터를 생성합니다."""
    rng = np.random.default_rng(42)
    start = pd.Timestamp.today().normalize() - pd.Timedelta(days=n_days)
    menus = ["Americano", "Latte", "Cappuccino", "Mocha", "Espresso", "Hot Chocolate"]
    rows = []
    for i in range(n_days):
        date = start + pd.Timedelta(days=i)
        weekday = date.dayofweek
        weekend_boost = 1.25 if weekday >= 5 else 1.0
        for menu in menus:
            base = {
                "Americano": 45, "Latte": 28, "Cappuccino": 18,
                "Mocha": 12, "Espresso": 10, "Hot Chocolate": 8
            }[menu]
            qty = max(0, int(rng.normal(base * weekend_boost, 6)))
            price = {
                "Americano": 4000, "Latte": 5000, "Cappuccino": 5200,
                "Mocha": 5500, "Espresso": 3500, "Hot Chocolate": 5300
            }[menu]
            rows.append({
                "date": date.date().isoformat(),
                "menu_name": menu,
                "qty": qty,
                "sales_amount": qty * price
            })
    return pd.DataFrame(rows)

def load_hf_coffee_dataset() -> pd.DataFrame:
    """허깅페이스 CoffeeSales 데이터셋을 로드합니다. 실패하면 샘플 데이터를 반환합니다."""
    try:
        from datasets import load_dataset
        ds = load_dataset(DATASET_NAME, split="train")
        df = ds.to_pandas()
        print(f"✅ Hugging Face 데이터셋 로드 성공: {DATASET_NAME}")
        print(f"원본 shape: {df.shape}")
        return df
    except Exception as e:
        print("⚠️ Hugging Face 데이터셋 로드 실패. 실습용 샘플 데이터를 생성합니다.")
        print("사유:", repr(e))
        return make_fallback_coffee_sales()

def find_col(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in columns:
        c_low = c.lower()
        if any(cand.lower() in c_low for cand in candidates):
            return c
    return None

def normalize_to_jangsa_schema(raw_df: pd.DataFrame) -> pd.DataFrame:
    """여러 형태의 커피 판매 데이터를 장사비서 표준 스키마로 변환합니다."""
    df = raw_df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    date_col = find_col(df.columns, ["date", "datetime", "timestamp", "transaction_date", "order_date"])
    menu_col = find_col(df.columns, ["coffee_name", "coffee", "product", "product_name", "menu", "menu_name", "item"])
    amount_col = find_col(df.columns, ["money", "sales_amount", "amount", "revenue", "price", "total", "sales"])
    qty_col = find_col(df.columns, ["qty", "quantity", "count", "unit_sold", "units_sold"])

    out = pd.DataFrame()

    if date_col:
        out["date"] = pd.to_datetime(df[date_col], errors="coerce").dt.date.astype("string")
    else:
        # 날짜가 없으면 거래 순서 기준으로 가상의 날짜를 부여합니다.
        base = pd.Timestamp.today().normalize() - pd.Timedelta(days=max(30, len(df)//10))
        out["date"] = [(base + pd.Timedelta(days=i//10)).date().isoformat() for i in range(len(df))]

    if menu_col:
        out["menu_name"] = df[menu_col].astype(str).str.strip()
    else:
        out["menu_name"] = "Coffee"

    if qty_col:
        out["qty"] = pd.to_numeric(df[qty_col], errors="coerce").fillna(1).clip(lower=0)
    else:
        # 거래 단위 데이터는 1잔 판매로 간주합니다.
        out["qty"] = 1

    if amount_col:
        out["sales_amount"] = pd.to_numeric(df[amount_col], errors="coerce").fillna(0).clip(lower=0)
    else:
        # 금액이 없으면 메뉴별 기본 가격을 가정합니다.
        price_map = {
            "Americano": 4000, "Latte": 5000, "Cappuccino": 5200,
            "Mocha": 5500, "Espresso": 3500, "Hot Chocolate": 5300
        }
        out["sales_amount"] = out["menu_name"].map(price_map).fillna(4500) * out["qty"]

    out = out.dropna(subset=["date"])
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.dropna(subset=["date"])

    # 일자/메뉴 단위로 집계: POS 업로드 데이터 형태에 맞춤
    out["date"] = out["date"].dt.date.astype("string")
    sales = (
        out.groupby(["date", "menu_name"], as_index=False)
           .agg(qty=("qty", "sum"), sales_amount=("sales_amount", "sum"))
    )
    sales["qty"] = sales["qty"].astype(float)
    sales["sales_amount"] = sales["sales_amount"].astype(float)
    return sales

raw_df = load_hf_coffee_dataset()
raw_path = "data/raw/hf_coffee_sales_raw.csv"
raw_df.to_csv(raw_path, index=False, encoding="utf-8-sig")

sales_df = normalize_to_jangsa_schema(raw_df)
processed_path = "data/processed/sales_cleaned.csv"
sales_df.to_csv(processed_path, index=False, encoding="utf-8-sig")

print("✅ 장사비서 표준 포맷 저장:", processed_path)
display(sales_df.head())
print(sales_df.describe(include="all"))


README.md:   0%|          | 0.00/924 [00:00<?, ?B/s]

index.csv:   0%|          | 0.00/201k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2623 [00:00<?, ? examples/s]

✅ Hugging Face 데이터셋 로드 성공: tablegpt/CoffeeSales
원본 shape: (2623, 6)
✅ 장사비서 표준 포맷 저장: data/processed/sales_cleaned.csv


,date,menu_name,qty,sales_amount
0,2024-03-01,Americano,1.0,28.9
1,2024-03-01,Americano with Milk,4.0,135.2
2,2024-03-01,Cocoa,1.0,38.7
3,2024-03-01,Hot Chocolate,3.0,116.1
4,2024-03-01,Latte,2.0,77.4


              date menu_name          qty  sales_amount
count         1270      1270  1270.000000   1270.000000
unique         294         8          NaN           NaN
top     2024-10-25     Latte          NaN           NaN
freq             8       240          NaN           NaN
mean           NaN       NaN     2.065354     65.863071
std            NaN       NaN     1.409071     46.685455
min            NaN       NaN     1.000000     18.120000
25%            NaN       NaN     1.000000     32.820000
50%            NaN       NaN     2.000000     51.920000
75%            NaN       NaN     3.000000     82.820000
max            NaN       NaN    12.000000    339.480000


### 2.4 에이전트 설계 개요

이번 노트북에서 에이전트는 모델 내부의 가중치를 직접 업데이트하지 않습니다.  
대신 아래 역할을 수행합니다.

```text
데이터 관찰 → 학습 가능 여부 판단 → 실험 실행 → 성능 평가 → 개선안 작성 → 재학습 조건 판단
```

즉, 에이전트는 **ML 학습 오케스트레이터**입니다.


> **이 셀이 정상 동작하면**, 서버 코드 작성으로 넘어갑니다.  
> 데이터셋 로드가 실패해도 샘플 데이터로 계속 진행되도록 설계되어 있습니다.


### 2.5 에이전트 정책 문서 생성

아래 셀은 에이전트가 참고할 정책 문서를 `agent/` 폴더에 생성합니다.

- `soul.md`: 에이전트의 정체성
- `data_policy.md`: 데이터 품질 기준
- `experiment_policy.md`: 실험 설계 기준
- `evaluation_policy.md`: 모델 평가 기준
- `retrain_policy.md`: 재학습 조건
- `rollback_policy.md`: 롤백 기준

In [4]:
from pathlib import Path

agent_docs = {
    "soul.md": """
# Jangsa ML Agent Soul

너는 장사비서 ML 학습 관리자다.

너의 목적은 모델을 무조건 학습시키는 것이 아니다.
좋은 데이터, 명확한 기준, 검증된 성능을 바탕으로
소상공인에게 도움이 되는 예측 모델을 안전하게 개선하는 것이다.

원칙:
1. 데이터가 부족하면 학습하지 않는다.
2. 없는 숫자를 만들지 않는다.
3. 모델 성능이 좋아졌다는 증거 없이는 교체하지 않는다.
4. 예측 결과와 실제 결과를 반드시 비교한다.
5. 모든 판단은 기록한다.
6. 사람의 승인 없이 운영 모델을 교체하지 않는다.
""",
    "data_policy.md": """
# Data Policy

필수 컬럼:
- date
- menu_name
- qty
- sales_amount

데이터 부족 기준:
- 14일 미만: ML 학습 금지
- 30일 미만: 참고 분석만 가능
- 90일 이상: 기본 예측 가능
- 365일 이상: 계절성 분석 가능

이상치:
- 매출 금액이 음수인 경우
- qty가 0인데 sales_amount가 있는 경우
- 날짜가 미래인 경우
- 동일 주문이 과도하게 중복된 경우
""",
    "experiment_policy.md": """
# Experiment Policy

에이전트는 한 번에 하나의 핵심 변경만 실험한다.

좋은 실험:
- 요일 피처 추가
- 최근 7일 평균 피처 추가
- 메뉴 카테고리 추가
- 날씨 피처 추가

나쁜 실험:
- 모델, 피처, 전처리를 동시에 모두 바꾸는 것
- 평가 기준 없이 학습하는 것
- 검증 데이터 없이 성능을 주장하는 것
""",
    "evaluation_policy.md": """
# Evaluation Policy

총매출 예측 지표:
- MAE
- RMSE
- MAPE

모델 교체 조건:
1. 기존 모델보다 주요 지표가 개선되어야 한다.
2. 특정 요일이나 메뉴에서 성능이 크게 나빠지면 안 된다.
3. 검증 데이터 성능이 좋아야 한다.
4. 에이전트가 개선 이유를 설명할 수 있어야 한다.
5. 사람이 승인해야 한다.
""",
    "retrain_policy.md": """
# Retrain Policy

재학습 제안 조건:
1. 신규 데이터가 30일 이상 추가됨
2. 최근 14일 예측 오차가 MAPE 20% 초과
3. 메뉴 구성이 크게 변경됨
4. 영업시간이 변경됨
5. 계절이 바뀜
6. 사용자가 재학습을 요청함
""",
    "rollback_policy.md": """
# Rollback Policy

새 모델 배포 후 다음 문제가 발생하면 이전 모델로 되돌린다.

1. 실제 운영 예측 오차가 기존보다 커짐
2. 특정 메뉴 예측이 비정상적으로 튐
3. 데이터 전처리 오류 발생
4. 사용자 리포트 품질 저하
5. API 응답 오류 증가
"""
}

for filename, content in agent_docs.items():
    path = Path("agent") / filename
    path.write_text(content.strip() + "\n", encoding="utf-8")

print("✅ 에이전트 정책 문서 생성 완료")
print("\n".join(str(p) for p in Path("agent").glob("*.md")))

✅ 에이전트 정책 문서 생성 완료
agent/experiment_policy.md
agent/data_policy.md
agent/rollback_policy.md
agent/evaluation_policy.md
agent/retrain_policy.md
agent/soul.md


### 2.6 에이전트 함수: 데이터 진단 + 학습 가능 여부 판단

아래 셀은 실제 LLM 호출 없이도 에이전트형 판단 루프를 테스트할 수 있도록 규칙 기반 에이전트 함수를 만듭니다.

In [5]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

def agent_data_diagnosis(df: pd.DataFrame) -> dict:
    required = ["date", "menu_name", "qty", "sales_amount"]
    missing_cols = [c for c in required if c not in df.columns]

    report = {
        "rows": int(len(df)),
        "missing_required_columns": missing_cols,
        "date_count": 0,
        "menu_count": 0,
        "negative_sales_rows": 0,
        "zero_qty_positive_sales_rows": 0,
        "status": "unknown",
        "messages": []
    }

    if missing_cols:
        report["status"] = "blocked"
        report["messages"].append(f"필수 컬럼 누락: {missing_cols}")
        return report

    tmp = df.copy()
    tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
    report["date_count"] = int(tmp["date"].dt.date.nunique())
    report["menu_count"] = int(tmp["menu_name"].nunique())
    report["negative_sales_rows"] = int((pd.to_numeric(tmp["sales_amount"], errors="coerce") < 0).sum())
    report["zero_qty_positive_sales_rows"] = int(((pd.to_numeric(tmp["qty"], errors="coerce") == 0) & (pd.to_numeric(tmp["sales_amount"], errors="coerce") > 0)).sum())

    if report["date_count"] < 14:
        report["status"] = "blocked"
        report["messages"].append("14일 미만 데이터입니다. ML 학습보다 패턴 리포트가 적합합니다.")
    elif report["date_count"] < 30:
        report["status"] = "caution"
        report["messages"].append("30일 미만 데이터입니다. 예측은 참고용으로만 사용하세요.")
    elif report["date_count"] < 90:
        report["status"] = "trainable_basic"
        report["messages"].append("기본 매출 예측 모델 학습이 가능합니다.")
    else:
        report["status"] = "trainable_good"
        report["messages"].append("요일성/최근 흐름을 반영한 기본 예측 모델 학습이 가능합니다.")

    if report["negative_sales_rows"] > 0:
        report["messages"].append("음수 매출이 있어 정제가 필요합니다.")
    if report["zero_qty_positive_sales_rows"] > 0:
        report["messages"].append("qty=0인데 매출이 있는 행이 있어 확인이 필요합니다.")

    return report

sales_df = pd.read_csv("data/processed/sales_cleaned.csv")
data_report = agent_data_diagnosis(sales_df)

Path("data/reports").mkdir(parents=True, exist_ok=True)
Path("data/reports/data_quality_report.json").write_text(
    json.dumps(data_report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(data_report, ensure_ascii=False, indent=2))

{
  "rows": 1270,
  "missing_required_columns": [],
  "date_count": 294,
  "menu_count": 8,
  "negative_sales_rows": 0,
  "zero_qty_positive_sales_rows": 0,
  "status": "trainable_good",
  "messages": [
    "요일성/최근 흐름을 반영한 기본 예측 모델 학습이 가능합니다."
  ]
}



---

## 3. 프로젝트 뼈대 코드

---


### 3.1 폴더 구조 — 장사비서 ML + 에이전트

In [6]:

import os

dirs = [
    "app",
    "models/current",
    "models/candidates",
    "models/archive",
    "frontend",
    "data/raw",
    "data/processed",
    "data/reports",
    "experiments",
    "ml_pipeline",
    "agent",
    "logs"
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("프로젝트 구조:")
print("""
jangsa-agent-ml/
├── 📁 app/
│   ├── auth.py                 ← API Key 인증
│   ├── schemas.py              ← 입력/출력 스키마
│   ├── model_service.py        ← 모델 로드 + 예측 + 에이전트 리포트
│   └── main.py                 ← FastAPI 서버
│
├── 📁 ml_pipeline/
│   ├── train_sales_model.py    ← 매출 예측 모델 학습
│   └── agent_report.py         ← 에이전트 리포트 생성
│
├── 📁 data/
│   ├── raw/                    ← 허깅페이스 원본 데이터
│   ├── processed/              ← 장사비서 표준 데이터
│   └── reports/                ← 데이터 품질 리포트
│
├── 📁 agent/
│   ├── soul.md
│   ├── data_policy.md
│   ├── experiment_policy.md
│   ├── evaluation_policy.md
│   ├── retrain_policy.md
│   └── rollback_policy.md
│
├── 📁 models/
│   ├── current/                ← 운영 모델
│   ├── candidates/             ← 후보 모델
│   └── archive/                ← 이전 모델
│
├── 📁 frontend/
│   └── app.py                  ← Streamlit UI
│
└── requirements.txt
""")


프로젝트 구조:

jangsa-agent-ml/
├── 📁 app/
│   ├── auth.py                 ← API Key 인증
│   ├── schemas.py              ← 입력/출력 스키마
│   ├── model_service.py        ← 모델 로드 + 예측 + 에이전트 리포트
│   └── main.py                 ← FastAPI 서버
│
├── 📁 ml_pipeline/
│   ├── train_sales_model.py    ← 매출 예측 모델 학습
│   └── agent_report.py         ← 에이전트 리포트 생성
│
├── 📁 data/
│   ├── raw/                    ← 허깅페이스 원본 데이터
│   ├── processed/              ← 장사비서 표준 데이터
│   └── reports/                ← 데이터 품질 리포트
│
├── 📁 agent/
│   ├── soul.md
│   ├── data_policy.md
│   ├── experiment_policy.md
│   ├── evaluation_policy.md
│   ├── retrain_policy.md
│   └── rollback_policy.md
│
├── 📁 models/
│   ├── current/                ← 운영 모델
│   ├── candidates/             ← 후보 모델
│   └── archive/                ← 이전 모델
│
├── 📁 frontend/
│   └── app.py                  ← Streamlit UI
│
└── requirements.txt



### 3.1.1 ml_pipeline/train_sales_model.py — 모델 학습 파이프라인

In [7]:
%%writefile ml_pipeline/train_sales_model.py
"""
장사비서 커피 매출 예측 모델 학습 파이프라인

입력:
- data/processed/sales_cleaned.csv

출력:
- models/current/sales_predictor.joblib
- models/current/metrics.json
- logs/training_log.md
"""

from pathlib import Path
import json
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1, 1, np.abs(y_true))
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100)


def make_daily_features(sales_df: pd.DataFrame):
    df = sales_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    daily = (
        df.groupby("date", as_index=False)
          .agg(
              total_sales=("sales_amount", "sum"),
              total_qty=("qty", "sum"),
              menu_count=("menu_name", "nunique")
          )
          .sort_values("date")
    )

    # 날짜가 빠진 경우에도 시계열 피처가 안정적으로 계산되도록 일 단위로 보정
    full_range = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
    daily = daily.set_index("date").reindex(full_range).rename_axis("date").reset_index()
    daily[["total_sales", "total_qty", "menu_count"]] = daily[["total_sales", "total_qty", "menu_count"]].fillna(0)

    daily["dayofweek"] = daily["date"].dt.dayofweek
    daily["month"] = daily["date"].dt.month
    daily["is_weekend"] = (daily["dayofweek"] >= 5).astype(int)
    daily["lag_1_sales"] = daily["total_sales"].shift(1)
    daily["rolling_7day_sales"] = daily["total_sales"].shift(1).rolling(7, min_periods=1).mean()
    daily["rolling_7day_qty"] = daily["total_qty"].shift(1).rolling(7, min_periods=1).mean()

    daily = daily.dropna().reset_index(drop=True)

    feature_cols = [
        "dayofweek",
        "month",
        "is_weekend",
        "lag_1_sales",
        "rolling_7day_sales",
        "rolling_7day_qty",
        "menu_count"
    ]
    return daily, feature_cols


def train(input_path="data/processed/sales_cleaned.csv", model_dir="models/current"):
    Path(model_dir).mkdir(parents=True, exist_ok=True)
    Path("logs").mkdir(exist_ok=True)

    sales_df = pd.read_csv(input_path)
    daily, feature_cols = make_daily_features(sales_df)

    if len(daily) < 14:
        raise ValueError("학습 가능한 일자 수가 14일 미만입니다. ML 학습보다 패턴 리포트가 적합합니다.")

    split_idx = max(1, int(len(daily) * 0.8))
    train_df = daily.iloc[:split_idx].copy()
    test_df = daily.iloc[split_idx:].copy()

    if len(test_df) < 3:
        # 데이터가 적을 때도 노트북 실습이 멈추지 않도록 마지막 3일을 테스트로 사용
        train_df = daily.iloc[:-3].copy()
        test_df = daily.iloc[-3:].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["total_sales"]
    X_test = test_df[feature_cols]
    y_test = test_df["total_sales"]

    baseline_pred = np.repeat(y_train.mean(), len(y_test))

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        min_samples_leaf=2
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    metrics = {
        "trained_at": datetime.now().isoformat(timespec="seconds"),
        "data_rows": int(len(sales_df)),
        "daily_rows": int(len(daily)),
        "train_days": int(len(train_df)),
        "test_days": int(len(test_df)),
        "features": feature_cols,
        "model": "RandomForestRegressor",
        "baseline_mae": float(mean_absolute_error(y_test, baseline_pred)),
        "baseline_rmse": float(mean_squared_error(y_test, baseline_pred) ** 0.5),
        "baseline_mape": mape(y_test, baseline_pred),
        "model_mae": float(mean_absolute_error(y_test, pred)),
        "model_rmse": float(mean_squared_error(y_test, pred) ** 0.5),
        "model_mape": mape(y_test, pred),
        "target": "daily_total_sales"
    }
    metrics["mape_improvement_point"] = float(metrics["baseline_mape"] - metrics["model_mape"])
    metrics["is_model_better_than_baseline"] = bool(metrics["model_mape"] < metrics["baseline_mape"])

    bundle = {
        "model": model,
        "feature_cols": feature_cols,
        "metrics": metrics,
        "last_known_sales": float(daily["total_sales"].iloc[-1]),
        "last_known_qty": float(daily["total_qty"].iloc[-1]),
        "last_7day_sales": float(daily["total_sales"].tail(7).mean()),
        "last_7day_qty": float(daily["total_qty"].tail(7).mean()),
        "last_menu_count": float(daily["menu_count"].tail(7).mean())
    }

    joblib.dump(bundle, Path(model_dir) / "sales_predictor.joblib")
    Path(model_dir, "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

    with open("logs/training_log.md", "a", encoding="utf-8") as f:
        f.write(f"## Training {metrics['trained_at']}\n\n")
        f.write(json.dumps(metrics, ensure_ascii=False, indent=2))
        f.write("\n\n")

    return metrics


if __name__ == "__main__":
    result = train()
    print(json.dumps(result, ensure_ascii=False, indent=2))

Writing ml_pipeline/train_sales_model.py


### 3.1.2 ml_pipeline/agent_report.py — 에이전트 학습 리포트 생성

In [8]:
%%writefile ml_pipeline/agent_report.py
"""
에이전트형 학습 리포트 생성기

입력:
- data/reports/data_quality_report.json
- models/current/metrics.json

출력:
- logs/agent_decisions.md
- dict 형태 리포트
"""

from pathlib import Path
import json
from datetime import datetime


def load_json(path, default=None):
    p = Path(path)
    if not p.exists():
        return default
    return json.loads(p.read_text(encoding="utf-8"))


def generate_agent_report(
    data_report_path="data/reports/data_quality_report.json",
    metrics_path="models/current/metrics.json"
):
    data_report = load_json(data_report_path, {})
    metrics = load_json(metrics_path, {})

    messages = []
    actions = []
    risk_level = "LOW"

    status = data_report.get("status", "unknown")
    date_count = data_report.get("date_count", 0)

    if status == "blocked":
        risk_level = "HIGH"
        messages.append("데이터가 부족하거나 필수 조건을 만족하지 않아 모델 학습을 권장하지 않습니다.")
        actions.append("최소 14일 이상의 매출 데이터를 확보하세요.")
    elif status == "caution":
        risk_level = "MEDIUM"
        messages.append("데이터가 적어 예측 결과는 참고용으로만 사용해야 합니다.")
        actions.append("30일 이상 데이터를 쌓은 뒤 재학습하세요.")
    else:
        messages.append("기본 매출 예측 모델 학습이 가능한 데이터 상태입니다.")

    if metrics:
        model_mape = metrics.get("model_mape")
        baseline_mape = metrics.get("baseline_mape")
        improvement = metrics.get("mape_improvement_point")
        better = metrics.get("is_model_better_than_baseline")

        if better:
            messages.append(f"모델 MAPE가 기준 모델보다 개선되었습니다. 개선폭: {improvement:.2f}%p")
            actions.append("운영 모델 후보로 등록할 수 있습니다. 단, 사람 승인 후 반영하세요.")
        else:
            risk_level = "MEDIUM" if risk_level == "LOW" else risk_level
            messages.append("새 모델이 기준 모델보다 뚜렷하게 개선되지 않았습니다.")
            actions.append("요일/시간대/프로모션/날씨 피처를 추가한 다음 실험을 권장합니다.")

        if model_mape is not None and model_mape > 25:
            risk_level = "MEDIUM"
            actions.append("예측 오차가 큽니다. 데이터 기간과 이상치를 다시 점검하세요.")

    if date_count < 90:
        actions.append("90일 이상 데이터가 쌓이면 요일성/월별 흐름을 더 안정적으로 반영할 수 있습니다.")

    actions = list(dict.fromkeys(actions))

    report = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "agent_role": "Agent ML Learning Orchestrator",
        "risk_level": risk_level,
        "data_status": status,
        "messages": messages,
        "recommended_actions": actions,
        "human_approval_required": True
    }

    Path("logs").mkdir(exist_ok=True)
    with open("logs/agent_decisions.md", "a", encoding="utf-8") as f:
        f.write(f"## Agent Report {report['generated_at']}\n\n")
        f.write(f"- Risk Level: {risk_level}\n")
        f.write(f"- Data Status: {status}\n\n")
        f.write("### 판단\n")
        for msg in messages:
            f.write(f"- {msg}\n")
        f.write("\n### 다음 액션\n")
        for action in actions:
            f.write(f"- {action}\n")
        f.write("\n---\n\n")

    return report


if __name__ == "__main__":
    print(json.dumps(generate_agent_report(), ensure_ascii=False, indent=2))

Writing ml_pipeline/agent_report.py


### 3.1.3 모델 학습 실행 + 에이전트 리포트 확인

In [9]:
# 모델 학습 실행
from ml_pipeline.train_sales_model import train
from ml_pipeline.agent_report import generate_agent_report

metrics = train()
agent_report = generate_agent_report()

print("✅ 모델 학습 지표")
print(json.dumps(metrics, ensure_ascii=False, indent=2))

print("\n✅ 에이전트 리포트")
print(json.dumps(agent_report, ensure_ascii=False, indent=2))

✅ 모델 학습 지표
{
  "trained_at": "2026-06-18T06:24:00",
  "data_rows": 1270,
  "daily_rows": 297,
  "train_days": 237,
  "test_days": 60,
  "features": [
    "dayofweek",
    "month",
    "is_weekend",
    "lag_1_sales",
    "rolling_7day_sales",
    "rolling_7day_qty",
    "menu_count"
  ],
  "model": "RandomForestRegressor",
  "baseline_mae": 106.93567651195498,
  "baseline_rmse": 137.10872926484134,
  "baseline_mape": 510.67544420688347,
  "model_mae": 71.80090966883115,
  "model_rmse": 92.70325879968304,
  "model_mape": 70.84560790843575,
  "target": "daily_total_sales",
  "mape_improvement_point": 439.82983629844773,
  "is_model_better_than_baseline": true
}

✅ 에이전트 리포트
{
  "generated_at": "2026-06-18T06:24:00",
  "agent_role": "Agent ML Learning Orchestrator",
  "risk_level": "MEDIUM",
  "data_status": "trainable_good",
  "messages": [
    "기본 매출 예측 모델 학습이 가능한 데이터 상태입니다.",
    "모델 MAPE가 기준 모델보다 개선되었습니다. 개선폭: 439.83%p"
  ],
  "recommended_actions": [
    "운영 모델 후보로 등록할 수 있습니다. 단, 사람 승

### 3.2 auth.py — 재사용

In [10]:
%%writefile app/auth.py
"""
Day 6에서 만든 인증 모듈을 그대로 재사용합니다.
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]

Writing app/auth.py


### 3.3 schemas.py — 장사비서 입력/출력 스키마

In [11]:

%%writefile app/schemas.py
"""
장사비서 매출 예측 API 스키마
"""

from typing import List, Optional
from pydantic import BaseModel, Field


class PredictRequest(BaseModel):
    predict_date: str = Field(..., description="예측할 날짜. 예: 2026-06-19")
    expected_menu_count: Optional[float] = Field(None, ge=0, description="예상 판매 메뉴 수. 없으면 최근 평균 사용")
    memo: Optional[str] = Field(None, max_length=1000, description="사장님 메모. 예: 비 예보, 근처 행사, 신메뉴 출시")


class PredictResponse(BaseModel):
    success: bool
    predict_date: str
    predicted_sales_amount: float
    confidence_note: str
    jangsa_actions: List[str]
    agent_messages: List[str]


class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    model_metrics: dict


class AgentReportResponse(BaseModel):
    success: bool
    report: dict


Writing app/schemas.py


### 3.4 model_service.py — 모델 로드 + 예측 + 에이전트 리포트

In [12]:

%%writefile app/model_service.py
"""
모델 로드, 매출 예측, 에이전트 리포트 함수를 정의합니다.
"""

from pathlib import Path
import json
import math

import joblib
import pandas as pd

MODEL_PATH = Path("models/current/sales_predictor.joblib")
METRICS_PATH = Path("models/current/metrics.json")
DATA_REPORT_PATH = Path("data/reports/data_quality_report.json")


def load_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def load_model():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            "모델 파일이 없습니다. 먼저 ml_pipeline/train_sales_model.py를 실행하세요."
        )
    return joblib.load(MODEL_PATH)


def make_features_for_date(model_bundle: dict, predict_date: str, expected_menu_count=None):
    dt = pd.to_datetime(predict_date)

    feature_row = {
        "dayofweek": int(dt.dayofweek),
        "month": int(dt.month),
        "is_weekend": int(dt.dayofweek >= 5),
        "lag_1_sales": float(model_bundle.get("last_known_sales", 0)),
        "rolling_7day_sales": float(model_bundle.get("last_7day_sales", 0)),
        "rolling_7day_qty": float(model_bundle.get("last_7day_qty", 0)),
        "menu_count": float(expected_menu_count if expected_menu_count is not None else model_bundle.get("last_menu_count", 1))
    }

    feature_cols = model_bundle["feature_cols"]
    return pd.DataFrame([[feature_row[c] for c in feature_cols]], columns=feature_cols)


def generate_jangsa_actions(predicted_sales: float, predict_date: str, memo: str = None):
    dt = pd.to_datetime(predict_date)
    actions = []

    if dt.dayofweek >= 5:
        actions.append("주말 예측일입니다. 디저트/라떼 계열의 재고를 평일보다 여유 있게 확인하세요.")
    else:
        actions.append("평일 예측일입니다. 출근/점심 시간대의 기본 메뉴 재고를 먼저 점검하세요.")

    if predicted_sales < 300000:
        actions.append("예측 매출이 낮은 편입니다. 오후 시간대 세트 메뉴나 재방문 쿠폰 테스트를 고려하세요.")
    elif predicted_sales > 600000:
        actions.append("예측 매출이 높은 편입니다. 피크 시간대 제조 동선과 원재료 부족 가능성을 점검하세요.")
    else:
        actions.append("예측 매출이 중간 수준입니다. 베스트셀러 중심으로 재고를 안정적으로 준비하세요.")

    if memo:
        actions.append(f"사장님 메모 반영 필요: {memo}")

    return actions[:3]


def predict(model_bundle: dict, predict_date: str, expected_menu_count=None, memo: str = None) -> dict:
    X = make_features_for_date(model_bundle, predict_date, expected_menu_count)
    y_pred = float(model_bundle["model"].predict(X)[0])
    y_pred = max(0.0, y_pred)

    metrics = model_bundle.get("metrics", {})
    model_mape = metrics.get("model_mape")

    if model_mape is None:
        confidence_note = "성능 지표가 없어 신뢰도 판단이 제한적입니다."
    elif model_mape <= 15:
        confidence_note = f"검증 MAPE {model_mape:.1f}%로 비교적 안정적인 예측입니다."
    elif model_mape <= 25:
        confidence_note = f"검증 MAPE {model_mape:.1f}%입니다. 참고용 예측으로 사용하세요."
    else:
        confidence_note = f"검증 MAPE {model_mape:.1f}%로 오차가 큰 편입니다. 데이터 보강이 필요합니다."

    data_report = load_json(DATA_REPORT_PATH, {})
    agent_messages = []
    if data_report.get("messages"):
        agent_messages.extend(data_report["messages"][:2])
    if metrics.get("is_model_better_than_baseline"):
        agent_messages.append("현재 모델은 기준 모델보다 개선된 후보입니다.")
    else:
        agent_messages.append("현재 모델은 기준 모델 대비 개선폭이 작을 수 있습니다.")

    return {
        "predicted_sales_amount": round(y_pred, 0),
        "confidence_note": confidence_note,
        "jangsa_actions": generate_jangsa_actions(y_pred, predict_date, memo),
        "agent_messages": agent_messages
    }


def get_metrics():
    return load_json(METRICS_PATH, {})


def get_agent_report():
    try:
        from ml_pipeline.agent_report import generate_agent_report
        return generate_agent_report()
    except Exception as e:
        return {
            "agent_role": "Agent ML Learning Orchestrator",
            "risk_level": "UNKNOWN",
            "messages": [f"에이전트 리포트 생성 실패: {repr(e)}"],
            "recommended_actions": ["모델 학습과 데이터 리포트 파일이 존재하는지 확인하세요."],
            "human_approval_required": True
        }


Writing app/model_service.py


### 3.5 main.py — FastAPI 서버

In [13]:

%%writefile app/main.py
"""
FastAPI 서버: 장사비서 매출 예측 + 에이전트 리포트
"""

import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException

from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse, HealthResponse, AgentReportResponse
from app.model_service import load_model, predict, get_metrics, get_agent_report

app = FastAPI(
    title="Agent Jangsa ML API",
    description="허깅페이스 커피 매출 데이터 기반 장사비서 ML 모델 서빙 API",
    version="0.1.0"
)

executor = ThreadPoolExecutor(max_workers=2)
model_bundle = None


@app.on_event("startup")
async def startup_event():
    global model_bundle
    model_bundle = load_model()


@app.get("/health", response_model=HealthResponse)
async def health():
    return {
        "status": "ok",
        "model_loaded": model_bundle is not None,
        "model_metrics": get_metrics()
    }


@app.post("/predict", response_model=PredictResponse)
async def predict_sales(
    request: PredictRequest,
    user: str = Depends(verify_api_key)
):
    if model_bundle is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    loop = asyncio.get_running_loop()
    try:
        result = await loop.run_in_executor(
            executor,
            lambda: predict(
                model_bundle,
                predict_date=request.predict_date,
                expected_menu_count=request.expected_menu_count,
                memo=request.memo
            )
        )
        return {
            "success": True,
            "predict_date": request.predict_date,
            **result
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"예측 실패: {repr(e)}")


@app.get("/agent/report", response_model=AgentReportResponse)
async def agent_report(user: str = Depends(verify_api_key)):
    return {
        "success": True,
        "report": get_agent_report()
    }


Writing app/main.py


### 3.6 frontend/app.py — Streamlit 테스트 UI

In [14]:

%%writefile frontend/app.py
"""
Streamlit 프론트엔드: 장사비서 매출 예측 테스트 UI

실행:
streamlit run frontend/app.py
"""

from datetime import date, timedelta
import requests
import streamlit as st

API_URL = "http://localhost:8002"

st.set_page_config(page_title="에이전트 장사비서 ML", layout="centered")
st.title("☕ 에이전트 장사비서 ML 테스트")
st.caption("커피 매출 데이터 기반 내일 매출 예측 + 에이전트 리포트")

api_key = st.sidebar.text_input("API Key", value="test-key-001", type="password")
headers = {"X-API-Key": api_key}

tab1, tab2 = st.tabs(["매출 예측", "에이전트 리포트"])

with tab1:
    predict_date = st.date_input("예측 날짜", value=date.today() + timedelta(days=1))
    expected_menu_count = st.number_input("예상 판매 메뉴 수", min_value=0.0, value=6.0, step=1.0)
    memo = st.text_area("사장님 메모", placeholder="예: 비 예보, 근처 행사, 신메뉴 출시 등")

    if st.button("예측 실행"):
        payload = {
            "predict_date": predict_date.isoformat(),
            "expected_menu_count": expected_menu_count,
            "memo": memo or None
        }
        res = requests.post(f"{API_URL}/predict", json=payload, headers=headers)
        if res.status_code == 200:
            data = res.json()
            st.metric("예측 매출", f"{data['predicted_sales_amount']:,.0f} 원")
            st.info(data["confidence_note"])
            st.subheader("내일 할 일 3개")
            for action in data["jangsa_actions"]:
                st.write(f"- {action}")
            st.subheader("에이전트 판단")
            for msg in data["agent_messages"]:
                st.write(f"- {msg}")
        else:
            st.error(res.text)

with tab2:
    if st.button("에이전트 리포트 불러오기"):
        res = requests.get(f"{API_URL}/agent/report", headers=headers)
        if res.status_code == 200:
            report = res.json()["report"]
            st.json(report)
        else:
            st.error(res.text)


Writing frontend/app.py


In [27]:
!pkill -f streamlit || true

^C


In [28]:
!streamlit run frontend/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --browser.gatherUsageStats false \
  > streamlit.log 2>&1 &

In [30]:
from google.colab import output

PORT = 8501
base_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")

if not base_url.endswith("/"):
    base_url += "/"

print("Streamlit Dashboard:", base_url)

Streamlit Dashboard: https://8501-m-s-kkb-usc1b2-2fssycd65svhp-b.us-central1-2.prod.colab.dev/


### 3.7 requirements.txt — 실행 패키지

In [15]:
%%writefile requirements.txt
fastapi
uvicorn
streamlit
requests
pandas
numpy
scikit-learn
joblib
datasets

Writing requirements.txt


---

## 4. 작업 시간

---

### 4.1 권장 순서



```
Step 1. 모델 선택 + 노트북에서 동작 확인 (섹션 2.3)
        → "이 모델이 내 입력에 대해 결과를 반환하는가?"

Step 2. schemas.py 작성
        → "입력과 출력의 형태를 정의"

Step 3. model_service.py 작성
        → "모델 로드 + 추론 함수"

Step 4. main.py 작성
        → "FastAPI 서버 조립"

Step 5. 서버 실행 + Swagger UI 테스트
        → "API가 동작하는가?"

Step 6. frontend/app.py 작성
        → "Streamlit UI 연결"
```



### 4.2 서버 실행 (Step 5에서 사용)

> ⚠️ **코드를 수정했는데 반영이 안 될 때 — 커널을 재시작하세요.**
>
> `app/main.py`, `app/model_service.py` 등을 고친 뒤 아래 셀을 다시 실행해도,
> 이미 메모리에 올라간 이전 코드가 남아 변경이 반영되지 않을 수 있습니다.
> 코드를 수정했다면 **커널 재시작(Kernel → Restart) 후 맨 위 셀부터 다시 실행**하세요.

In [16]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main:app", port=8002)

서버 실행됨: http://127.0.0.1:8002


In [23]:
from google.colab import output

PORT = 8002

base_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")

# 끝에 /가 없으면 붙여주기
if not base_url.endswith("/"):
    base_url += "/"

print("Base URL:", base_url)
print("Swagger UI:", base_url + "docs")
print("Health Check:", base_url + "health")
print("Agent Report:", base_url + "agent/report")

Base URL: https://8002-m-s-kkb-usc1b2-2fssycd65svhp-b.us-central1-2.prod.colab.dev/
Swagger UI: https://8002-m-s-kkb-usc1b2-2fssycd65svhp-b.us-central1-2.prod.colab.dev/docs
Health Check: https://8002-m-s-kkb-usc1b2-2fssycd65svhp-b.us-central1-2.prod.colab.dev/health
Agent Report: https://8002-m-s-kkb-usc1b2-2fssycd65svhp-b.us-central1-2.prod.colab.dev/agent/report


In [17]:
import requests
import json

API_URL = "http://127.0.0.1:8002"

print("======== 1. Health Check ========")
try:
    r = requests.get(f"{API_URL}/health", timeout=5)
    print("status_code:", r.status_code)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))
except Exception as e:
    print("Health Check 실패:", e)

print("\n======== 2. Predict API ========")
payload = {
    "predict_date": "2024-12-20",
    "expected_menu_count": 8,
    "owner_memo": "비 오는 금요일이라 오후 매출이 걱정됩니다."
}

try:
    r = requests.post(f"{API_URL}/predict", json=payload, timeout=5)
    print("status_code:", r.status_code)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))
except Exception as e:
    print("Predict API 실패:", e)

print("\n======== 3. Agent Report ========")
try:
    r = requests.get(f"{API_URL}/agent/report", timeout=5)
    print("status_code:", r.status_code)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))
except Exception as e:
    print("Agent Report 실패:", e)

======== 1. Health Check ========
status_code: 200
{
  "status": "ok",
  "model_loaded": true,
  "model_metrics": {
    "trained_at": "2026-06-18T06:24:00",
    "data_rows": 1270,
    "daily_rows": 297,
    "train_days": 237,
    "test_days": 60,
    "features": [
      "dayofweek",
      "month",
      "is_weekend",
      "lag_1_sales",
      "rolling_7day_sales",
      "rolling_7day_qty",
      "menu_count"
    ],
    "model": "RandomForestRegressor",
    "baseline_mae": 106.93567651195498,
    "baseline_rmse": 137.10872926484134,
    "baseline_mape": 510.67544420688347,
    "model_mae": 71.80090966883115,
    "model_rmse": 92.70325879968304,
    "model_mape": 70.84560790843575,
    "target": "daily_total_sales",
    "mape_improvement_point": 439.82983629844773,
    "is_model_better_than_baseline": true
  }
}

======== 2. Predict API ========
status_code: 401
{
  "detail": "API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요."
}

======== 3. Agent Report ========
status_code: 401
{
  "detail": "AP

In [18]:
!python -m pip show jupyter-server-proxy

In [19]:
import os
from IPython.display import display, HTML

prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
print("JUPYTERHUB_SERVICE_PREFIX =", prefix)

port = 8002

display(HTML(f"""
<h3>자동 감지된 Jupyter Proxy 링크</h3>

<p><b>base prefix:</b> {prefix}</p>

<ul>
  <li>
    <a href="{prefix}proxy/{port}/docs" target="_blank">
      Swagger UI 열기
    </a>
  </li>
  <li>
    <a href="{prefix}proxy/{port}/health" target="_blank">
      Health Check 열기
    </a>
  </li>
  <li>
    <a href="{prefix}proxy/{port}/agent/report" target="_blank">
      Agent Report 열기
    </a>
  </li>
</ul>
"""))

JUPYTERHUB_SERVICE_PREFIX = /


In [20]:
from IPython.display import display, HTML

display(HTML("""
<h3>장사비서 API 데모 링크</h3>

<p>아래 링크를 클릭해서 Swagger UI 또는 Health Check를 열어보세요.</p>

<ul>
  <li>
    <a href="/user/07757bd0-e263-4293-b60d-d5ce3b3a77ce/nv-4017-r-18266/proxy/8002/docs" target="_blank">
      Swagger UI 열기
    </a>
  </li>
  <li>
    <a href="/user/07757bd0-e263-4293-b60d-d5ce3b3a77ce/nv-4017-r-18266/proxy/8002/health" target="_blank">
      Health Check 열기
    </a>
  </li>
  <li>
    <a href="/user/07757bd0-e263-4293-b60d-d5ce3b3a77ce/nv-4017-r-18266/proxy/8002/agent/report" target="_blank">
      Agent Report 열기
    </a>
  </li>
</ul>
"""))

### 4.3 API 테스트 템플릿

아래 테스트는 세 가지를 확인합니다.

1. `/health`: 모델 로드와 성능 지표 확인
2. `/predict`: 특정 날짜의 예상 매출과 내일 할 일 3개 확인
3. `/agent/report`: 에이전트의 데이터/모델 판단 리포트 확인


In [21]:

import requests
from datetime import date, timedelta
import json

API_URL = "http://localhost:8002"
HEADERS = {"X-API-Key": "test-key-001"}

# 1) health check
health = requests.get(f"{API_URL}/health").json()
print("✅ HEALTH")
print(json.dumps(health, ensure_ascii=False, indent=2))

# 2) 예측 테스트
payload = {
    "predict_date": (date.today() + timedelta(days=1)).isoformat(),
    "expected_menu_count": 6,
    "memo": "내일은 평소보다 오후 유입이 낮을 수 있음"
}

response = requests.post(
    f"{API_URL}/predict",
    json=payload,
    headers=HEADERS,
)
print("\n✅ PREDICT")
print(f"상태: {response.status_code}")
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

# 3) 에이전트 리포트
agent_response = requests.get(
    f"{API_URL}/agent/report",
    headers=HEADERS,
)
print("\n✅ AGENT REPORT")
print(f"상태: {agent_response.status_code}")
print(json.dumps(agent_response.json(), ensure_ascii=False, indent=2))


✅ HEALTH
{
  "status": "ok",
  "model_loaded": true,
  "model_metrics": {
    "trained_at": "2026-06-18T06:24:00",
    "data_rows": 1270,
    "daily_rows": 297,
    "train_days": 237,
    "test_days": 60,
    "features": [
      "dayofweek",
      "month",
      "is_weekend",
      "lag_1_sales",
      "rolling_7day_sales",
      "rolling_7day_qty",
      "menu_count"
    ],
    "model": "RandomForestRegressor",
    "baseline_mae": 106.93567651195498,
    "baseline_rmse": 137.10872926484134,
    "baseline_mape": 510.67544420688347,
    "model_mae": 71.80090966883115,
    "model_rmse": 92.70325879968304,
    "model_mape": 70.84560790843575,
    "target": "daily_total_sales",
    "mape_improvement_point": 439.82983629844773,
    "is_model_better_than_baseline": true
  }
}

✅ PREDICT
상태: 200
{
  "success": true,
  "predict_date": "2026-06-19",
  "predicted_sales_amount": 437.0,
  "confidence_note": "검증 MAPE 70.8%로 오차가 큰 편입니다. 데이터 보강이 필요합니다.",
  "jangsa_actions": [
    "평일 예측일입니다. 출근/점심 시간

### 4.4 막혔을 때 참고할 Day



```
"스키마를 어떻게 정의하지?"        → Day 2 섹션 4, Day 5 섹션 3
"FastAPI 서버 구조가 기억 안 나"   → Day 5 섹션 3, Day 6 섹션 6
"run_in_executor 사용법?"         → Day 3 섹션 4
"인증 적용 방법?"                  → Day 6 섹션 2
"Streamlit에서 API 호출?"         → Day 4 섹션 6, Day 5 섹션 4
"에러 처리?"                      → Day 3 섹션 5
```

---

## 5. 발표 및 회고

---

### 5.1 발표 (개인당 5분)

```
발표 항목:
  1. 어떤 도메인/태스크를 선택했는가?
  2. 어떤 모델을 사용했는가? (선택 이유)
  3. 데모 시연 (Swagger UI 또는 Streamlit)
  4. 구현하면서 가장 어려웠던 부분은?
```

### 5.2 회고

```
스스로 돌아보기:
  - Day 1~7 교안 없이 코드를 작성할 수 있었는가?
  - 어떤 부분에서 교안을 다시 찾아봤는가?
  - 다음에 다시 만든다면 무엇을 다르게 하겠는가?
```

---

## 6. 8일간의 여정 정리

---



### 6.1 Day 1의 문제 → Day 8의 해결

```
Day 1의 문제                           해결한 Day
──────────────────────────────        ──────────
라이브러리가 없음                       Day 1: requirements.txt
모델 구조 코드 필요                     Day 1: model_utils.py 모듈 분리
전처리 로직 누락                       Day 5/7: 전처리 파라미터 저장
비개발자가 사용할 수 없음               Day 4/5/7: Streamlit UI
누구나 API 호출 가능                   Day 6: API Key 인증
스스로 서비스를 만들 수 있는가?          Day 8: 자율 프로젝트 ✅
```



### 6.2 8일간 배운 기술 전체 지도

```
Day 1: 환경 세팅 + 모델 직렬화          "모델을 저장하고 불러온다"
Day 2: FastAPI + Pydantic              "모델을 API로 감싼다"
Day 3: 비동기 처리 + 에러/로깅          "안정적으로 돌아가게 한다"
Day 4: Streamlit + 시스템 아키텍처      "누구나 쓸 수 있게 한다"
Day 5: [프로젝트 1] 정형 데이터 서비스   "따라하며 조립한다"
Day 6: 인증 + 파일 업로드               "보안과 비정형 데이터를 다룬다"
Day 7: [프로젝트 2] 텍스트/이미지 서비스  "패턴을 반복하며 익힌다"
Day 8: [자율 프로젝트] 나만의 서비스      "스스로 만든다"
```



### 6.3 Next Step: MLOps로 가는 길

```
이 과정에서 배운 것:                 다음 과정에서 배울 것:
──────────────────                  ──────────────────
수동으로 서버 실행                    → Docker로 패키징
단일 서버에서 실행                    → 클라우드 배포 (AWS, GCP)
코드 변경 시 수동 재시작              → CI/CD 파이프라인 (자동 빌드/배포)
모델 버전 1개                        → 모델 버전 관리 (MLflow, DVC)
수동 모니터링 (로그 확인)             → 자동 모니터링 (Prometheus, Grafana)
```



> **"코드를 고칠 때마다 매번 서버를 재시작해야 하나요?"**
>
> 그 질문의 답이 MLOps입니다.
> CI/CD 파이프라인이 코드 변경을 감지하면 자동으로 빌드, 테스트, 배포합니다.
> 여러분은 코드를 커밋하기만 하면 됩니다.

---


### ✅ Day 8 최종 체크포인트

```
Q1. 이 프로젝트의 입력과 출력은 각각 무엇입니까?

Q2. Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?

Q3. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?

Q4. run_in_executor를 사용한 이유는 무엇입니까?

Q5. 에이전트는 모델 학습 과정에서 어떤 역할을 합니까?

Q6. 이번 모델을 운영 모델로 교체하기 전에 확인해야 할 기준은 무엇입니까?

Q7. 이 서비스를 실제 장사비서로 배포하려면 추가로 어떤 데이터가 필요합니까?
```


---

### 📌 Day 8 요약 & 전체 과정 완료

```
오늘 한 일:
  ✅ Day 1~7의 기술을 조합하여 나만의 서비스를 직접 만들었습니다.
  ✅ 교안 없이 설계 → 구현 → 테스트를 경험했습니다.
  ✅ 8일간의 여정을 회고하고, MLOps로 가는 길을 확인했습니다.

8일간의 전체 성과:
  🎉 PyTorch / HuggingFace 모델을 API로 서빙할 수 있습니다.
  🎉 비개발자도 사용 가능한 웹 UI를 붙일 수 있습니다.
  🎉 인증, 에러 처리, 로깅으로 안정적인 서비스를 만들 수 있습니다.
  🎉 스스로 설계하고 구현할 수 있다는 자신감을 얻었습니다.
```

### 제출

다음 내역을 MD 파일로 기록하고, 깃허브에 업로드하여 링크로 제출하시기 바랍니다.

1. 프로젝트 실행 내역 캡처와 설명
2. `/health` 테스트 결과
3. `/predict` 테스트 결과
4. `/agent/report` 테스트 결과
5. 에이전트 정책 문서 요약
6. 모델 성능 지표 요약
7. Day 8 최종 체크포인트 답변
8. 프로젝트 회고


다음 내역을 MD 파일로 기록, 깃헙에 업로드하여 링크로 제출하시기 바랍니다  

1. 프로젝트 실행 내역 캡쳐와 설명
2. 각 섹션 체크포인트의 답변
3. 프로젝트 회고

수고하셨습니다!

## 프로젝트 실행 내역

### 1. 프로젝트 구조 확인
프로젝트는 agent, app, data, experiments, frontend, logs, ml_pipeline, models 폴더로 구성했다.  
agent 폴더에는 에이전트 정책 문서를, ml_pipeline에는 데이터 검증·전처리·학습·평가 흐름을, app에는 FastAPI 서버 코드를 배치했다.

### 2. 데이터 로드 및 전처리
허깅페이스 커피 매출 데이터셋을 불러와 장사비서 표준 포맷인 date, menu_name, qty, sales_amount 컬럼으로 변환했다.  
최종 데이터는 1,270건이며, 일 단위 집계 기준 297일치 데이터로 구성되었다.

### 3. 모델 학습
RandomForestRegressor를 사용해 일매출 예측 모델을 학습했다.  
기준 모델 대비 MAE와 RMSE가 개선되었고, 모델은 baseline보다 나은 성능을 보였다.  
다만 MAPE가 아직 높기 때문에 실전 운영 전 추가 데이터 검증과 모델 개선이 필요하다.

### 4. FastAPI 서버 실행
학습된 모델을 FastAPI 서버에 연결하고 8002 포트에서 실행했다.  
Colab proxy를 통해 Swagger UI에 접속하여 API 문서를 확인했다.

### 5. Swagger UI 테스트
/health API를 통해 모델 로드 상태를 확인했고, /predict API를 통해 특정 날짜의 예상 매출과 내일 실행 액션을 확인했다.  
/agent/report API를 통해 에이전트가 모델 성능, 위험도, 사람 승인 필요 여부를 판단하는 리포트를 반환하는 것을 확인했다.